# Deterministic command replay

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

PKG_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PKG_ROOT))

from pipeline.opendata import (
    route_dataset_dir,
    accepted_command_flight_ids,
    operational_phases,
    DEFAULT_OPERATIONAL_PHASE_KW,
    phase_seconds_from_commands,
)
from pipeline.replay import replay_metrics, rollout_vertical_dynamics

ROUTE_NAME = 'EHAM_LSZH'
ROUTE_DIR = route_dataset_dir(ROUTE_NAME)
COMMANDS_DIR = ROUTE_DIR / 'commands'
ADSB_DIR = ROUTE_DIR / 'data' / 'adsb'
MODES_DIR = ROUTE_DIR / 'data' / 'modes_decoded'

FLIGHT_IDS = accepted_command_flight_ids(ROUTE_NAME)
REPLAY_KW = dict(tau_vz_s=0.0, start_phase='CLIMB')
print(len(FLIGHT_IDS), 'flights with accepted commands')

300 flights with accepted commands


In [2]:
rows = []
for fid in FLIGHT_IDS:
    cmd_path = COMMANDS_DIR / f'{fid}.parquet'
    if not cmd_path.exists():
        continue
    cmds = pd.read_parquet(cmd_path)
    rep = rollout_vertical_dynamics(cmds, **REPLAY_KW)
    adsb = pd.read_parquet(ADSB_DIR / f'{fid}.parquet') if (ADSB_DIR / f'{fid}.parquet').exists() else None
    modes = pd.read_parquet(MODES_DIR / f'{fid}.parquet') if (MODES_DIR / f'{fid}.parquet').exists() else None
    rows.append({'flight_id': fid, **replay_metrics(rep, adsb=adsb, modes=modes)})
summary = pd.DataFrame(rows)
sample = pd.concat(
    pd.read_parquet(COMMANDS_DIR / f'{fid}.parquet', columns=['vz_sel'])
    for fid in FLIGHT_IDS[:5]
)
print(f'vz_sel NaN fraction (should be ~0): {sample["vz_sel"].isna().mean():.4f}')
display(summary.describe())
print(f"median MAE h: {summary['mae_ft'].median():.0f} ft ({summary['mae_ft'].median() * 0.3048:.0f} m)")
print(f"median MAE γ: {summary['mae_gamma_deg'].median():.2f}° (gen vz & gen TAS vs obs, both from replay)")
print(f"median MAE TAS: {summary['mae_tas_kt'].median():.1f} kt (gen TAS from mach_sel/cas_sel vs Mode S)")

vz_sel NaN fraction (should be ~0): 0.1241


,rmse_ft,mae_ft,bias_ft,n,mae_gamma_deg,rmse_gamma_deg,n_gamma,mae_tas_kt,rmse_tas_kt,n_tas
count,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000
mean,2108.830275,1979.329766,536.907257,3977.700000,0.372592,0.740106,449.416667,13.011042,19.427357,449.416667
std,1609.469019,1600.836292,2459.653217,285.626466,0.098943,0.242834,53.376837,7.260462,8.545647,53.376837
min,193.156059,155.906502,-15159.849851,3512.000000,0.178726,0.331260,320.000000,5.491398,9.400148,320.000000
25%,949.519329,811.832533,-847.090704,3767.750000,0.307245,0.580510,414.000000,8.910763,14.715304,414.000000
50%,1695.904265,1551.761049,540.422037,3939.500000,0.359485,0.694040,446.500000,11.832668,18.049997,446.500000
75%,2986.371135,2896.727626,2433.475675,4120.000000,0.416481,0.845431,485.000000,14.998992,21.858179,485.000000
max,15426.944163,15159.922171,5448.202440,5470.000000,0.906035,1.829130,635.000000,84.568651,103.140132,635.000000


median MAE h: 1552 ft (473 m)
median MAE γ: 0.36° (gen vz & gen TAS vs obs, both from replay)
median MAE TAS: 11.8 kt (gen TAS from mach_sel/cas_sel vs Mode S)


In [ ]:
PHASE_COLOR = {
    'CLIMB': '#43A047', 'LEVEL': '#1E88E5', 'DESCENT': '#FB8C00',
    'DESCENT': '#E53935', 'GROUND': '#616161', 'NA': '#BDBDBD',
}


def plot_replay(replay: pd.DataFrame, flight_id: str, modes: pd.DataFrame | None = None):
    t = replay['t_s']
    if 'phase' in replay.columns:
        print('operational phase [s]:', phase_seconds_from_commands(replay))

    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=('Altitude', 'Vertical rate', 'TAS'),
                        row_heights=[0.5, 0.25, 0.25])

    if 'phase' in replay.columns:
        ta = t.to_numpy()
        ph = replay['phase'].astype(str).str.upper().to_numpy()
        seen = set()
        i = 0
        while i < len(ta):
            p, j = ph[i], i + 1
            while j < len(ta) and ph[j] == p:
                j += 1
            t1 = ta[j] if j < len(ta) else ta[-1] + 1
            c = PHASE_COLOR.get(p, PHASE_COLOR['NA'])
            for xref, yref in (('x', 'y domain'), ('x2', 'y2 domain'), ('x3', 'y3 domain')):
                fig.add_shape(type='rect', x0=ta[i], x1=t1, y0=0, y1=1,
                              xref=xref, yref=yref, fillcolor=c, opacity=0.28,
                              layer='below', line_width=0)
            seen.add(p)
            i = j
        for p in ('CLIMB', 'LEVEL', 'DESCENT', 'GROUND'):
            if p in seen:
                fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                    marker=dict(size=12, color=PHASE_COLOR[p], symbol='square'),
                    name=p.title(), showlegend=True), row=1, col=1)

    fig.add_trace(go.Scatter(x=t, y=replay['obs_altitude_ft'], mode='lines', name='observed', line=dict(color='gray')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=replay['gen_altitude_ft'], mode='lines', name='replay', line=dict(color='red')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=replay['cmd_alt_ft'], mode='lines', name='h_sel', line=dict(color='blue', dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=replay['obs_vertical_rate_fpm'], mode='lines', name='obs VS', line=dict(color='gray')), row=2, col=1)
    fig.add_trace(go.Scatter(x=t, y=replay['gen_rocd_fpm'], mode='lines', name='replay ROCD', line=dict(color='red')), row=2, col=1)
    vz_col = next((c for c in ('cmd_vz_sel_fpm', 'cmd_vz_target_fpm', 'cmd_vz_fpm') if c in replay.columns), None)
    if vz_col:
        fig.add_trace(go.Scatter(x=t, y=replay[vz_col], mode='lines', name='vz_sel', line=dict(color='blue', dash='dash')), row=2, col=1)
    if 'gen_tas_kt' in replay.columns:
        fig.add_trace(go.Scatter(
            x=t, y=replay['gen_tas_kt'], mode='lines', name='replay TAS',
            line=dict(color='red', width=2),
        ), row=3, col=1)
        if modes is not None and not modes.empty and 'TAS' in modes.columns:
            mo = modes.sort_values('timestamp').reset_index(drop=True)
            mo['timestamp'] = pd.to_datetime(mo['timestamp'], utc=True)
            merged = pd.merge_asof(
                replay.sort_values('timestamp'),
                mo[['timestamp', 'TAS']],
                on='timestamp',
                direction='nearest',
                tolerance=pd.Timedelta('2s'),
            )
            fig.add_trace(go.Scatter(
                x=t, y=merged['TAS'], mode='lines+markers', name='obs TAS (Mode S)',
                line=dict(color='gray', width=1), marker=dict(size=3, color='gray'),
            ), row=3, col=1)
    m = replay_metrics(replay)
    title = f"{flight_id} | RMSE h {m['rmse_ft']:.0f} ft"
    if m.get('n_gamma', 0):
        title += f" | MAE γ {m['mae_gamma_deg']:.2f}°"
    if m.get('n_tas', 0):
        title += f" | MAE TAS {m['mae_tas_kt']:.1f} kt"
    fig.update_layout(height=900, title=title, hovermode='x unified')
    fig.update_yaxes(title_text='ft', row=1, col=1)
    fig.update_yaxes(title_text='fpm', row=2, col=1)
    fig.update_yaxes(title_text='kt', row=3, col=1)
    fig.update_xaxes(title_text='time [s]', row=3, col=1)
    display(fig)

FLIGHT_INDEX = 0
output = widgets.Output()
prev_btn = widgets.Button(description='← Prev')
next_btn = widgets.Button(description='Next →')
label = widgets.HTML()

def render():
    global FLIGHT_INDEX
    with output:
        clear_output(wait=True)
        fid = FLIGHT_IDS[FLIGHT_INDEX]
        cmds = pd.read_parquet(COMMANDS_DIR / f'{fid}.parquet')
        modes_path = MODES_DIR / f'{fid}.parquet'
        modes = pd.read_parquet(modes_path) if modes_path.exists() else None
        adsb_path = ADSB_DIR / f'{fid}.parquet'
        adsb = pd.read_parquet(adsb_path) if adsb_path.exists() else None
        replay = rollout_vertical_dynamics(cmds, **REPLAY_KW)
        if 'phase' not in replay.columns:
            i0 = int(replay['replay_start_idx'].iloc[0]) if 'replay_start_idx' in replay.columns else 0
            if 'phase' in cmds.columns:
                replay['phase'] = cmds['phase'].iloc[i0 : i0 + len(replay)].to_numpy()
            else:
                replay['phase'] = operational_phases(
                    replay['obs_altitude_ft'], replay['obs_vertical_rate_fpm'], **DEFAULT_OPERATIONAL_PHASE_KW
                )
        label.value = f'<b>{FLIGHT_INDEX + 1}</b> / {len(FLIGHT_IDS)}: {fid}'
        if adsb is not None:
            m = replay_metrics(replay, adsb=adsb, modes=modes)
            print(f"MAE h {m['mae_ft']:.0f} ft | MAE γ {m['mae_gamma_deg']:.2f}° | MAE TAS {m['mae_tas_kt']:.1f} kt")
        plot_replay(replay, fid, modes=modes)


def go_prev(_):
    global FLIGHT_INDEX
    FLIGHT_INDEX = (FLIGHT_INDEX - 1) % len(FLIGHT_IDS)
    render()


def go_next(_):
    global FLIGHT_INDEX
    FLIGHT_INDEX = (FLIGHT_INDEX + 1) % len(FLIGHT_IDS)
    render()


prev_btn.on_click(go_prev)
next_btn.on_click(go_next)
display(widgets.HBox([prev_btn, next_btn, label]))
display(output)
render()

Output()